# 🎵 音乐转 MIDI / Music to MIDI（Colab GPU）

此 Colab 与桌面版、Space 使用同一套七模式工作流：

- **SMART 与四种钢琴模式**：点击主按钮后直接转换并下载一个 MIDI，不显示多音轨控件。
- **人声/伴奏分离、六声部分离**：主按钮只生成 2/6 个 WAV，不会自动转 MIDI，也不会自动合并 MIDI。
- 分离完成后显示真实波形音轨；每条音轨可单独勾选、从 **10 种转写路线**中选择模型，再明确点击该轨的“开始转换”。选择或滚动控件本身不会触发转换。
- 可把其他音频复制到本次会话目录并加入时间线；每次只转换被明确启动的单条音轨。

The Colab, desktop, and Space builds share the same seven-mode workflow. SMART and four piano modes convert directly. The two split modes create WAV files only, then expose real waveform rows for explicit, independent per-track MIDI conversion using one of 10 routes.


In [ ]:
#@title 1️⃣ 检查 GPU 可用性（详细日志）
import subprocess
import sys
from datetime import datetime

import torch


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


log("Starting runtime diagnostics")
log(f"Python version: {sys.version.split()[0]}")
log(f"PyTorch version: {torch.__version__}")
log(f"Torch CUDA build: {torch.version.cuda}")
log(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_index)
    total_vram_gb = props.total_memory / 1024**3
    log(f"Detected GPU: {props.name}")
    log(f"Total VRAM: {total_vram_gb:.2f} GB")
    log(f"SM count: {props.multi_processor_count}")
    try:
        reserved = torch.cuda.memory_reserved(device_index) / 1024**3
        allocated = torch.cuda.memory_allocated(device_index) / 1024**3
        log(f"Allocated VRAM: {allocated:.3f} GB")
        log(f"Reserved VRAM: {reserved:.3f} GB")
    except Exception as exc:
        log(f"Failed to read VRAM stats: {exc}")

    log("nvidia-smi summary:")
    try:
        smi = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used,driver_version",
                "--format=csv,noheader",
            ],
            text=True,
        ).strip()
        print(smi, flush=True)
    except Exception as exc:
        log(f"Unable to run nvidia-smi: {exc}")

    log("GPU runtime check passed")
else:
    log("No GPU detected. In Colab use Runtime -> Change runtime type -> T4 GPU")
    raise RuntimeError("GPU runtime is required")


In [ ]:
#@title 2️⃣ 克隆仓库并安装依赖（详细日志）
import importlib
import os
import shlex
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path


def log(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


def run_cmd(command, cwd=None):
    log(f"CMD: {command}")
    start = time.time()
    proc = subprocess.Popen(
        command,
        cwd=cwd,
        shell=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    line_count = 0
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
        line_count += 1
    return_code = proc.wait()
    elapsed = time.time() - start
    log(f"command finished: exit={return_code}, lines={line_count}, elapsed={elapsed:.1f}s")
    if return_code != 0:
        raise RuntimeError(f"command failed: {command}")


repo_dir = Path("/content/music-to-midi")
repo_url = "https://github.com/mason369/music-to-midi.git"

log("准备项目工作区")
if not repo_dir.exists():
    run_cmd(f"git clone --branch master --single-branch {repo_url} {repo_dir}")
else:
    log("仓库已存在，严格检查工作区并快进同步 origin/master")
    run_cmd("git -C /content/music-to-midi remote -v")
    dirty_output = subprocess.check_output(
        ["git", "-C", str(repo_dir), "status", "--porcelain"],
        text=True,
    ).strip()
    if dirty_output:
        raise RuntimeError(
            "Existing /content/music-to-midi worktree is dirty; refusing to overwrite:\n"
            + dirty_output
        )
    run_cmd("git -C /content/music-to-midi fetch origin master --prune")
    run_cmd("git -C /content/music-to-midi pull --ff-only origin master")

run_cmd("git -C /content/music-to-midi fetch origin master --prune")
head_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
origin_master_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "origin/master"], text=True
).strip()
if head_commit != origin_master_commit:
    raise RuntimeError(
        f"Colab source identity mismatch: HEAD={head_commit}, origin/master={origin_master_commit}"
    )
log(f"仓库源码已对齐 origin/master: {head_commit}")

os.chdir(repo_dir)
log(f"工作目录: {os.getcwd()}")

log("升级 pip/setuptools/wheel")
run_cmd("python -m pip install --upgrade pip setuptools wheel")

# ── 记录 Colab 预装的 torch 版本（绝不重装） ──
log("检测 Colab 预装 torch 版本")
import torch
preinstalled_torch_version = torch.__version__
log(f"torch=={torch.__version__}")
log(f"CUDA available: {torch.cuda.is_available()}, CUDA version: {torch.version.cuda}")

log("安装 Python 依赖")
# 保留 Colab 预装 torch；只安装当前 Web 版需要的依赖
packages = [
    "lightning",
    "einops",
    "transformers==4.48.3",
    "mir-eval",
    "deprecated",
    "smart-open",
    "nest-asyncio",
    "scipy",
    "scikit-learn",
    "torchmetrics",
    "wandb",
    "pretty-midi>=0.2.10",
    "soxr>=0.3.7",
    "huggingface_hub",
    "mido",
    "librosa",
    "soundfile",
    "transkun==2.0.1",
    "ariautils @ https://github.com/EleutherAI/aria-utils/archive/93da092204e5b1189ed8e0259f6156266fd086a7.zip",
    "safetensors>=0.4,<1",
    "orjson>=3.9,<4",
    "piano-transcription-inference==0.0.6",
    "torchlibrosa>=0.1.0,<0.2",
    "matplotlib>=3.7.0,<4",
    "gradio==4.44.1",
    "fastapi==0.115.2",
    "starlette==0.40.0",
    "tqdm",
    "pyyaml",
    "psutil",
    "numba",
]
quoted_packages = " ".join(shlex.quote(pkg) for pkg in packages)
run_cmd("python -m pip install " + quoted_packages)
aria_amt_requirement = "aria-amt @ https://github.com/EleutherAI/aria-amt/archive/a1ab73fc901d1759ec3bc173c146b3c6a3040261.zip"
run_cmd("python -m pip install --no-deps --force-reinstall " + shlex.quote(aria_amt_requirement))
from src.core.aria_amt_transcriber import get_aria_amt_runtime_unavailable_reason
aria_amt_identity_error = get_aria_amt_runtime_unavailable_reason()
if aria_amt_identity_error:
    raise RuntimeError(aria_amt_identity_error)
log("Aria-AMT source identity verified at a1ab73fc901d1759ec3bc173c146b3c6a3040261")

log("安装 audio-separator 运行依赖（固定兼容 NumPy 1.26）")
audio_separator_runtime_packages = [
    "numpy==1.26.4",
    "beartype==0.18.5",
    "diffq-fixed==0.2.4",
    "julius==0.2.7",
    "ml_collections==1.1.0",
    "onnx-weekly==1.21.0.dev20260302",
    "onnx2torch-py313==1.6.0",
    "pydub==0.25.1",
    "requests>=2.32.5,<3",
    "chardet>=5,<6",
    "onnxruntime-gpu==1.23.2",
    "resampy==0.4.3",
    "rotary-embedding-torch==0.6.5",
    "samplerate==0.1.0",
    "h5py>=3.10,<4",
    "mirdata>=0.3.8,<1",
    "six==1.17.0",
]
quoted_audio_separator_runtime_packages = " ".join(shlex.quote(pkg) for pkg in audio_separator_runtime_packages)
run_cmd("python -m pip install " + quoted_audio_separator_runtime_packages)
run_cmd("python -m pip install " + shlex.quote("audio-separator==0.44.1") + " --no-deps")
installed_torch_version = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__)"],
    text=True,
).strip()
if installed_torch_version != preinstalled_torch_version:
    raise RuntimeError(
        f"Dependency installation replaced Colab torch: {preinstalled_torch_version} -> {installed_torch_version}"
    )
log(f"Colab torch preserved: {installed_torch_version}")

log("使用 apt 安装 FFmpeg")
run_cmd("apt-get update")
run_cmd("apt-get install -y ffmpeg")

log("关键包版本")
for module_name in ["torch", "torchaudio", "torchvision", "gradio", "huggingface_hub", "lightning", "librosa"]:
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "unknown")
        log(f"{module_name}=={version}")
    except Exception as exc:
        log(f"无法读取 {module_name} 版本: {exc}")

log("依赖安装完成")

In [ ]:
#@title 3️⃣ 同步 YourMT3+ 源码（模型权重将在转换时按需准备）
import os
import sys
from datetime import datetime
from pathlib import Path

os.chdir("/content/music-to-midi")


def log(message=""):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}", flush=True)


yourmt3_dir = Path("/content/music-to-midi/YourMT3")
amt_src = yourmt3_dir / "amt" / "src"
entrypoint = amt_src / "model" / "ymt3.py"
if not entrypoint.is_file():
    raise RuntimeError(f"项目内置 YourMT3 补丁源码不完整: {entrypoint}")

sys.path.insert(0, "/content/music-to-midi")
from src.utils.yourmt3_source_identity import validate_patched_yourmt3_source

source_sha256, source_file_count = validate_patched_yourmt3_source(amt_src)
log(f"YourMT3 项目补丁源码就绪: files={source_file_count}, manifest_sha256={source_sha256}")
if amt_src not in sys.path:
    sys.path.insert(0, str(amt_src))
    log("已将 amt/src 添加到 sys.path")
log("YourMT3+ 源码已就绪；YourMT3+/MIROS、分离器与钢琴 checkpoint 将在点击转换后按所选路线下载并严格校验")

In [ ]:
#@title 4️⃣ 启动 Gradio 应用（详细日志 + 公开链接）
import atexit
import logging
import os
import platform
import shutil
import subprocess
import sys
import tempfile
import threading
import uuid
from datetime import datetime
from pathlib import Path


def tlog(message=""):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {message}", flush=True)


os.chdir("/content/music-to-midi")
try:
    subprocess.run(["fuser", "-k", "7860/tcp"], capture_output=True, text=True, timeout=5)
except Exception as exc:
    tlog(f"Skip port cleanup: {exc}")

PROJECT_ROOT = "/content/music-to-midi"
amt_src = "/content/music-to-midi/YourMT3/amt/src"
for p in [PROJECT_ROOT, amt_src]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_MIN_LOG_LEVEL"] = "3"
os.environ["NUMBA_CACHE_DIR"] = "/tmp/numba_cache"
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
logger = logging.getLogger("colab-app")

import gradio as gr
import torch

logger.info("========== Startup Diagnostics ==========")
logger.info(f"Python version: {platform.python_version()}")
logger.info(f"Gradio version: {gr.__version__}")
logger.info(f"Torch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")
logger.info(f"Project root: {PROJECT_ROOT}")
logger.info(f"YourMT3 source path: {amt_src}")
logger.info(f"YourMT3 ymt3.py exists: {Path(amt_src, 'model', 'ymt3.py').exists()}")

try:
    from model.ymt3 import YourMT3  # noqa: F401
    logger.info("✅ YourMT3 code import success")
except ImportError as exc:
    raise RuntimeError(f"YourMT3 import failed: {exc}, 请重新运行第 3 步")

from src.models.data_models import Config, ProcessingStage
from src.core.pipeline import MusicToMidiPipeline
from src.core.manual_midi import (
    MANUAL_MIDI_ROUTES,
    MIDI_ROUTE_MIROS,
    MIDI_ROUTE_PIANO_ARIA_AMT,
    MIDI_ROUTE_PIANO_BYTEDANCE_PEDAL,
    MIDI_ROUTE_PIANO_TRANSKUN,
    MIDI_ROUTE_PIANO_TRANSKUN_V2_AUG,
    MIDI_ROUTE_YOURMT3_PREFIX,
    build_manual_midi_config,
    manual_midi_output_dir,
)
from src.core.multi_stem_separator import STEM_KEYS
from src.core.separation_service import AudioSeparationService, SeparationResult
from src.utils.gpu_utils import clear_gpu_memory
from src.utils.yourmt3_downloader import DEFAULT_MODEL, OFFICIAL_YOURMT3_MODEL_KEYS, YOURMT3_MODELS
from src.gui.web.track_mixer_runtime import (
    TRACK_COLORS as MIXER_TRACK_COLORS,
    build_track_mixer_html,
    display_track_name,
    mixer_head,
)
from src.i18n.translator import Translator

device_label = f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"
logger.info(f"Current device: {device_label}")

model_cache_root = Path(os.path.expanduser("~/.cache/music_ai_models/yourmt3_all"))
ckpts = sorted(model_cache_root.rglob("*.ckpt")) if model_cache_root.exists() else []
logger.info(f"Model cache root: {model_cache_root}")
logger.info(f"Checkpoint count: {len(ckpts)}")

YOURMT3_MODEL_LABEL_TO_KEY = {
    YOURMT3_MODELS[model_key].get("ui_label", model_key): model_key
    for model_key in OFFICIAL_YOURMT3_MODEL_KEYS
}
YOURMT3_MODEL_CHOICES = list(YOURMT3_MODEL_LABEL_TO_KEY.keys())
DEFAULT_YOURMT3_MODEL_LABEL = next(
    label for label, model_key in YOURMT3_MODEL_LABEL_TO_KEY.items() if model_key == DEFAULT_MODEL
)
MULTI_INSTRUMENT_PROCESSING_MODES = {"smart"}
SPLIT_PROCESSING_MODES = {"vocal_split", "six_stem_split"}
DIRECT_PROCESSING_MODES = frozenset({
    "smart", "piano_transkun", "piano_transkun_v2_aug",
    "piano_aria_amt", "piano_bytedance_pedal",
})
SUPPORTED_AUDIO_SUFFIXES = frozenset({".mp3", ".wav", ".flac", ".ogg", ".m4a"})
MAX_MANUAL_TRACK_ROWS = 12
MULTI_BACKEND_LABEL_TO_KEY = {"YourMT3+": "yourmt3", "MIROS": "miros"}
MULTI_BACKEND_CHOICES = list(MULTI_BACKEND_LABEL_TO_KEY.keys())
DEFAULT_MULTI_BACKEND_LABEL = "YourMT3+"
MANUAL_MIDI_ROUTE_LABEL_TO_KEY = {}
for _route in MANUAL_MIDI_ROUTES:
    if _route.startswith(MIDI_ROUTE_YOURMT3_PREFIX):
        _model_key = _route.removeprefix(MIDI_ROUTE_YOURMT3_PREFIX)
        _route_label = f"YourMT3+ · {YOURMT3_MODELS[_model_key].get('ui_label', _model_key)}"
    else:
        _route_label = {
            MIDI_ROUTE_MIROS: "MIROS · multi-instrument",
            MIDI_ROUTE_PIANO_TRANSKUN: "TransKun V2 · piano",
            MIDI_ROUTE_PIANO_TRANSKUN_V2_AUG: "TransKun V2 Aug · piano",
            MIDI_ROUTE_PIANO_ARIA_AMT: "Aria-AMT · piano",
            MIDI_ROUTE_PIANO_BYTEDANCE_PEDAL: "ByteDance Pedal · piano",
        }[_route]
    MANUAL_MIDI_ROUTE_LABEL_TO_KEY[_route_label] = _route
MANUAL_MIDI_ROUTE_CHOICES = [(label, route) for label, route in MANUAL_MIDI_ROUTE_LABEL_TO_KEY.items()]
if len(MANUAL_MIDI_ROUTE_CHOICES) != 10 or len(MANUAL_MIDI_ROUTE_CHOICES) != len(MANUAL_MIDI_ROUTES):
    raise RuntimeError("Manual MIDI route UI must expose exactly the 10 shared routes")
DEFAULT_MANUAL_MIDI_ROUTE = MANUAL_MIDI_ROUTES[0]


def _manual_route_display_label(route):
    for label, key in MANUAL_MIDI_ROUTE_LABEL_TO_KEY.items():
        if key == route:
            return label
    raise gr.Error(ct("error.unknown_manual_route", route=route))
COLAB_LANGUAGE = os.environ.get("MUSIC_TO_MIDI_LANGUAGE", "zh_CN")
if COLAB_LANGUAGE not in Translator.AVAILABLE_LANGUAGES:
    raise RuntimeError(f"Unsupported MUSIC_TO_MIDI_LANGUAGE: {COLAB_LANGUAGE}")
COLAB_TRANSLATOR = Translator(COLAB_LANGUAGE)
COLAB_I18N = {
    "zh_CN": {
        "error.upload_required": "请先上传音频文件",
        "error.unknown_mode": "未知处理模式: {mode}",
        "error.unknown_yourmt3_model": "未知 YourMT3+ 模型模式: {model}",
        "error.unknown_backend": "未知多乐器转写后端: {backend}",
        "error.conversion_failed": "处理失败: {error}",
        "stage.preprocessing": "预处理",
        "stage.separation": "音源分离",
        "stage.transcription": "音频转写",
        "stage.vocal_transcription": "人声转写",
        "stage.synthesis": "MIDI合成",
        "stage.complete": "完成",
        "status.complete_header": "--- 转换完成 ---",
        "status.elapsed": "耗时",
        "status.seconds": "秒",
        "status.total_notes": "总音符数",
        "status.device": "设备",
        "status.output_count": "输出文件数",
        "status.midi_file": "MIDI 文件",
        "yourmt3.model_title": "YourMT3+ 模型模式",
        "yourmt3.checkpoint": "检查点",
        "yourmt3.traits": "适合/特点",
        "output.title": "输出说明",
        "output.single_midi": "直接转换并输出一个 MIDI 文件；不会显示多音轨控件。",
        "mode_info.smart": "**SMART** — 通过当前选择的 {backend_route} 后端对完整混音进行多乐器转写，识别 {instrument_standard} 乐器并直接生成 MIDI。",
        "mode_info.vocal_split": "**VOCAL_SPLIT** — 使用 {separator_chain} 分离原混音，只生成 vocals/accompaniment WAV。完成后在每条波形音轨上独立决定是否转 MIDI、选择模型并点击开始。",
        "mode_info.six_stem_split": "**六声部分离** — 使用 {separator_model} 分离 {stem_names} 六个 WAV。完成后在每条波形音轨上独立决定是否转 MIDI、选择模型并点击开始。",
        "mode_info.piano_transkun": "**{model_name}** — 钢琴专用转写，直接输出 MIDI，不显示多音轨控件。",
        "mode_info.piano_transkun_v2_aug": "**{model_name}** — 使用官方 {checkpoint_name} 数据增强路线，直接输出 MIDI。",
        "mode_info.piano_aria_amt": "**{model_name}** — 钢琴专用转写，首次使用会检查 {checkpoint_name}，直接输出 MIDI。",
        "mode_info.piano_bytedance_pedal": "**{model_name}** — 钢琴专用带踏板转写，保留延音踏板 {pedal_cc}，直接输出 MIDI。",
        "ui.header": "# 🎵 Music to MIDI\n七种处理模式保持同一工作流：SMART 与四种钢琴模式直接转换；人声/伴奏和六声部分离只生成 WAV，再在波形音轨中逐条选择是否转 MIDI。",
        "ui.audio_hint": "支持 MP3, WAV, FLAC, OGG, M4A（自动转换为 WAV 处理）",
        "ui.mode_label": "处理模式",
        "ui.backend_info": "仅 SMART 使用；分离模式在完成后的每条音轨上独立选择模型",
        "ui.yourmt3_model_info": "仅 SMART + YourMT3+ 使用；分离音轨在各自下拉框中选择",
        "ui.device": "当前设备",
        "ui.status": "状态",
        "ui.status_placeholder": "等待处理...",
        "ui.footer": "<center>基于 [YourMT3+](https://github.com/mimbres/YourMT3) 官方模型模式 | [GitHub](https://github.com/mason369/music-to-midi)</center>",
        "ui.launching": "🚀 启动中... 公开链接将在下方显示（详细日志模式）",
        "error.invalid_track_state": "音轨状态无效: {error}",
        "error.track_not_enabled": "请先勾选此音轨的‘转 MIDI’",
        "error.unknown_manual_route": "未知的逐轨转写模型: {route}",
        "error.add_audio_required": "请先选择要添加的音频文件",
        "error.too_many_tracks": "最多显示 {limit} 条音轨",
        "status.audio_added": "已添加 {count} 条音轨。",
        "output.vocal_manual": "主按钮只分离并保留 vocals/accompaniment 两个 WAV；不会自动生成 MIDI。",
        "output.six_stem_manual": "主按钮只分离并保留六个 stem WAV；不会自动生成 MIDI 或合并 MIDI。",
        "output.manual_tracks": "分离后，每条音轨可独立勾选、选择 10 种转写路线之一，并明确点击开始转换。",
        "ui.track_enable": "转 MIDI",
        "ui.track_start": "开始转换",
        "ui.track_status": "音轨状态",
        "ui.track_download": "下载此音轨 MIDI"
    },
    "en_US": {
        "error.upload_required": "Please upload an audio file first",
        "error.unknown_mode": "Unknown processing mode: {mode}",
        "error.unknown_yourmt3_model": "Unknown YourMT3+ model mode: {model}",
        "error.unknown_backend": "Unknown multi-instrument transcription backend: {backend}",
        "error.conversion_failed": "Processing failed: {error}",
        "stage.preprocessing": "Preprocessing",
        "stage.separation": "Separation",
        "stage.transcription": "Transcription",
        "stage.vocal_transcription": "Vocal transcription",
        "stage.synthesis": "MIDI synthesis",
        "stage.complete": "Complete",
        "status.complete_header": "--- Conversion Complete ---",
        "status.elapsed": "Elapsed",
        "status.seconds": "seconds",
        "status.total_notes": "Total notes",
        "status.device": "Device",
        "status.output_count": "Output file count",
        "status.midi_file": "MIDI file",
        "yourmt3.model_title": "YourMT3+ model mode",
        "yourmt3.checkpoint": "Checkpoint",
        "yourmt3.traits": "Best for / traits",
        "output.title": "Output notes",
        "output.single_midi": "Converts directly to one MIDI file and does not display the multi-track controls.",
        "mode_info.smart": "**SMART** — Uses the selected {backend_route} backend for full-mix multi-instrument transcription, recognizes {instrument_standard} instruments, and directly generates MIDI.",
        "mode_info.vocal_split": "**VOCAL_SPLIT** — Uses {separator_chain} on the original mix and generates only vocals/accompaniment WAV files. Then decide whether to transcribe each waveform track, select its model, and click Start.",
        "mode_info.six_stem_split": "**Six-stem separation** — Uses {separator_model} to produce {stem_names} WAV files. Then decide whether to transcribe each waveform track, select its model, and click Start.",
        "mode_info.piano_transkun": "**{model_name}** — Piano transcription that directly outputs MIDI without the multi-track controls.",
        "mode_info.piano_transkun_v2_aug": "**{model_name}** — Uses the official augmented {checkpoint_name} route and directly outputs MIDI.",
        "mode_info.piano_aria_amt": "**{model_name}** — Piano transcription that checks {checkpoint_name} on first use and directly outputs MIDI.",
        "mode_info.piano_bytedance_pedal": "**{model_name}** — Pedal-aware piano transcription preserving sustain {pedal_cc} and directly outputting MIDI.",
        "ui.header": "# 🎵 Music to MIDI\nThe same seven-mode workflow is used on every platform: SMART and four piano modes convert directly; vocal/two-stem and six-stem modes only create WAV files, followed by explicit per-track MIDI conversion from waveform rows.",
        "ui.audio_hint": "Supports MP3, WAV, FLAC, OGG, M4A (automatically converted to WAV for processing)",
        "ui.mode_label": "Processing mode",
        "ui.backend_info": "Used only by SMART; split modes choose a model independently on each resulting track",
        "ui.yourmt3_model_info": "Used only by SMART + YourMT3+; split tracks select routes in their own rows",
        "ui.device": "Current device",
        "ui.status": "Status",
        "ui.status_placeholder": "Waiting for processing...",
        "ui.footer": "<center>Powered by official [YourMT3+](https://github.com/mimbres/YourMT3) model modes | [GitHub](https://github.com/mason369/music-to-midi)</center>",
        "ui.launching": "🚀 Launching... The public link will appear below (verbose logging mode)",
        "error.invalid_track_state": "Invalid track state: {error}",
        "error.track_not_enabled": "Enable 'Convert to MIDI' for this track first",
        "error.unknown_manual_route": "Unknown per-track transcription model: {route}",
        "error.add_audio_required": "Select audio files to add first",
        "error.too_many_tracks": "At most {limit} tracks can be displayed",
        "status.audio_added": "Added {count} track(s).",
        "output.vocal_manual": "The main button only separates and keeps vocals/accompaniment WAV files; it does not generate MIDI automatically.",
        "output.six_stem_manual": "The main button only separates and keeps six stem WAV files; it does not generate stem or merged MIDI automatically.",
        "output.manual_tracks": "After separation, each track can be enabled independently, assigned one of 10 routes, and converted only by an explicit Start click.",
        "ui.track_enable": "Convert to MIDI",
        "ui.track_start": "Start conversion",
        "ui.track_status": "Track status",
        "ui.track_download": "Download this track's MIDI"
    }
}


def ct(key, **kwargs):
    text = COLAB_I18N[COLAB_LANGUAGE][key]
    return text.format(**kwargs) if kwargs else text


COLAB_MODE_IDS = (
    "smart",
    "vocal_split",
    "six_stem_split",
    "piano_transkun",
    "piano_transkun_v2_aug",
    "piano_aria_amt",
    "piano_bytedance_pedal",
)
COLAB_MODE_CHOICES = [(COLAB_TRANSLATOR.t(f"main.mode.{mode_id}"), mode_id) for mode_id in COLAB_MODE_IDS]
COLAB_BACKEND_ROUTE_NAMES = {"yourmt3": "YourMT3+", "miros": "MIROS"}
COLAB_MODE_INFO_TERMS = {
    "smart": {"instrument_standard": "GM"},
    "vocal_split": {"separator_chain": "Leap XE 90-band vocals + BS-PolarFormer 62-band accompaniment"},
    "six_stem_split": {"separator_model": "BS-RoFormer SW Fixed", "stem_names": "bass/drums/guitar/piano/vocals/other"},
    "piano_transkun": {"model_name": "TransKun V2 (default)"},
    "piano_transkun_v2_aug": {"model_name": "TransKun V2 Aug", "checkpoint_name": "checkpointTransformerAug / checkpointMSimplerAug"},
    "piano_aria_amt": {"model_name": "Aria-AMT", "checkpoint_name": "Aria-AMT checkpoint"},
    "piano_bytedance_pedal": {"model_name": "ByteDance Pedal", "pedal_cc": "CC64"},
}


def uses_yourmt3_processing_mode(mode, backend_key):
    return mode in MULTI_INSTRUMENT_PROCESSING_MODES and backend_key == "yourmt3"


LOG_FILE = "/tmp/midi_process.log"
COLAB_OUTPUT_ROOT = Path(tempfile.mkdtemp(prefix="music_to_midi_colab_session_"))


def _cleanup_colab_output_root():
    try:
        if COLAB_OUTPUT_ROOT.exists():
            shutil.rmtree(COLAB_OUTPUT_ROOT)
    except OSError as exc:
        logger.error(f"Unable to remove Colab output session at shutdown: {exc}")


atexit.register(_cleanup_colab_output_root)


class _RobustFileHandler(logging.Handler):
    def __init__(self, filename):
        super().__init__()
        self.filename = filename

    def emit(self, record):
        try:
            msg = self.format(record)
            with open(self.filename, "a", encoding="utf-8") as f:
                f.write(msg + "\n")
        except Exception:
            pass


_fh = _RobustFileHandler(LOG_FILE)
_fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S"))
_fh.setLevel(logging.INFO)
for _name in ("colab-app", "src.core", "src.utils"):
    logging.getLogger(_name).addHandler(_fh)


def clear_logs():
    try:
        open(LOG_FILE, "w", encoding="utf-8").close()
    except Exception:
        pass


def read_logs():
    try:
        with open(LOG_FILE, "r", encoding="utf-8", errors="replace") as f:
            content = f.read().replace("\x00", "")
        lines = content.strip().split("\n")
        return "\n".join(lines[-200:]) if lines and lines[0] else ""
    except Exception as exc:
        return f"[error] {exc}"


def ensure_aria_amt_weights():
    from download_aria_amt_model import download_aria_model, is_aria_model_available

    if is_aria_model_available():
        logger.info("Aria-AMT checkpoint found")
        return
    logger.info("Aria-AMT checkpoint not found, downloading via download_aria_amt_model.py")
    download_aria_model(printer=logger.info)


# === Colab runtime contract: direct modes + WAV-only split + explicit per-track MIDI ===

def resolve_multi_backend(backend_label):
    backend_key = MULTI_BACKEND_LABEL_TO_KEY.get(backend_label)
    if backend_key is None:
        raise gr.Error(ct("error.unknown_backend", backend=backend_label))
    return backend_key


def prepare_selected_models(mode, backend_key, yourmt3_model_key):
    if mode == "smart":
        if backend_key == "yourmt3":
            from src.utils.yourmt3_downloader import download_model
            logger.info(f"Preparing selected YourMT3+ checkpoint: {yourmt3_model_key}")
            download_model(yourmt3_model_key, progress_callback=lambda _p, message: logger.info(message))
        elif backend_key == "miros":
            from download_miros_model import prepare_miros_model
            logger.info("Preparing selected MIROS backend and weights")
            prepare_miros_model(printer=logger.info)
        else:
            raise ValueError(f"Unsupported multi-instrument backend: {backend_key!r}")
    if mode == "vocal_split":
        from download_accompaniment_model import download_accompaniment_model
        from download_vocal_model import download_vocal_model
        logger.info("Preparing Leap XE 90-band vocals assets")
        download_vocal_model(printer=logger.info)
        logger.info("Preparing BS-PolarFormer 62-band accompaniment assets")
        download_accompaniment_model(printer=logger.info)
    elif mode == "six_stem_split":
        from download_multistem_model import download_multistem_model
        logger.info("Preparing BS-RoFormer SW Fixed six-stem assets")
        download_multistem_model(printer=logger.info)
    elif mode == "piano_transkun_v2_aug":
        from download_transkun_v2_aug_model import download_transkun_v2_aug_model
        logger.info("Preparing official TransKun V2 Aug checkpoint")
        download_transkun_v2_aug_model(printer=logger.info)
    elif mode == "piano_aria_amt":
        ensure_aria_amt_weights()
    elif mode == "piano_bytedance_pedal":
        from download_bytedance_piano_model import download_bytedance_piano_model
        download_bytedance_piano_model(printer=logger.info)


def prepare_manual_midi_route(route):
    if route not in MANUAL_MIDI_ROUTES:
        raise gr.Error(ct("error.unknown_manual_route", route=route))
    config = build_manual_midi_config(Config(), route)
    prepare_selected_models(config.processing_mode, config.transcription_backend, config.yourmt3_model)
    return config


def _make_progress_callback(progress):
    def on_progress(p):
        stage_name = {
            ProcessingStage.PREPROCESSING: ct("stage.preprocessing"),
            ProcessingStage.SEPARATION: ct("stage.separation"),
            ProcessingStage.TRANSCRIPTION: ct("stage.transcription"),
            ProcessingStage.VOCAL_TRANSCRIPTION: ct("stage.vocal_transcription"),
            ProcessingStage.SYNTHESIS: ct("stage.synthesis"),
            ProcessingStage.COMPLETE: ct("stage.complete"),
        }.get(p.stage, str(p.stage))
        logger.info(f"Progress {p.overall_progress * 100:5.1f}% [{stage_name}] {p.message}")
        progress(p.overall_progress, desc=f"[{stage_name}] {p.message}")
    return on_progress


def _require_source_audio(path_value):
    if not path_value:
        raise gr.Error(ct("error.upload_required"))
    path = Path(path_value).resolve()
    if path.suffix.lower() not in SUPPORTED_AUDIO_SUFFIXES:
        raise ValueError(f"Unsupported audio extension: {path.suffix}")
    if not path.is_file() or path.stat().st_size <= 0:
        raise ValueError(f"Audio file is missing or empty: {path}")
    return path


def _require_active_request_dir(path_value):
    if not path_value:
        raise ValueError("Missing Colab request directory")
    session_root = COLAB_OUTPUT_ROOT.resolve()
    request_dir = Path(path_value).resolve()
    if request_dir == session_root or request_dir.parent != session_root or not request_dir.name.startswith("request-"):
        raise ValueError(f"Path is not an active Colab request directory: {request_dir}")
    if not request_dir.is_dir():
        raise ValueError(f"Colab request directory does not exist: {request_dir}")
    return request_dir


def _require_owned_request_file(request_dir_value, path_value, label, suffixes=None):
    request_dir = _require_active_request_dir(request_dir_value)
    path = Path(path_value).resolve()
    try:
        path.relative_to(request_dir)
    except ValueError as exc:
        raise ValueError(f"{label} is outside the active request: {path}") from exc
    if not path.is_file() or path.stat().st_size <= 0:
        raise ValueError(f"{label} is missing or empty: {path}")
    if suffixes is not None and path.suffix.lower() not in suffixes:
        raise ValueError(f"{label} has an unsupported extension: {path.suffix}")
    return path


def _require_owned_request_output_dir(request_dir_value, path_value):
    request_dir = _require_active_request_dir(request_dir_value)
    path = Path(path_value).resolve()
    try:
        relative = path.relative_to(request_dir)
    except ValueError as exc:
        raise ValueError(f"MIDI output directory is outside the active request: {path}") from exc
    if not relative.parts or relative.parts[0] == "..":
        raise ValueError(f"Invalid MIDI output directory: {path}")
    path.mkdir(parents=True, exist_ok=True)
    return path


def _normalize_track_state(state):
    if not isinstance(state, dict):
        raise gr.Error(ct("error.invalid_track_state", error="state must be a mapping"))
    mode = state.get("mode")
    if mode not in SPLIT_PROCESSING_MODES:
        raise gr.Error(ct("error.invalid_track_state", error=f"unsupported mode {mode!r}"))
    try:
        request_dir = _require_active_request_dir(state.get("request_dir"))
    except ValueError as exc:
        raise gr.Error(ct("error.invalid_track_state", error=exc)) from exc
    tracks = state.get("tracks")
    if not isinstance(tracks, list) or len(tracks) > MAX_MANUAL_TRACK_ROWS:
        raise gr.Error(ct("error.invalid_track_state", error="invalid track list length"))
    normalized_tracks = []
    seen_ids = set()
    for raw_track in tracks:
        if not isinstance(raw_track, dict):
            raise gr.Error(ct("error.invalid_track_state", error="track must be a mapping"))
        track_id = str(raw_track.get("id", ""))
        if not track_id or track_id in seen_ids:
            raise gr.Error(ct("error.invalid_track_state", error="duplicate track id"))
        seen_ids.add(track_id)
        try:
            audio_path = _require_owned_request_file(request_dir, raw_track.get("audio_path"), f"track {track_id} audio", SUPPORTED_AUDIO_SUFFIXES)
        except ValueError as exc:
            raise gr.Error(ct("error.invalid_track_state", error=exc)) from exc
        route = str(raw_track.get("route", DEFAULT_MANUAL_MIDI_ROUTE))
        if route not in MANUAL_MIDI_ROUTES:
            raise gr.Error(ct("error.unknown_manual_route", route=route))
        midi_path_value = raw_track.get("midi_path")
        midi_path = None
        if midi_path_value:
            try:
                midi_path = str(_require_owned_request_file(request_dir, midi_path_value, f"track {track_id} MIDI", {".mid", ".midi"}))
            except ValueError as exc:
                raise gr.Error(ct("error.invalid_track_state", error=exc)) from exc
        normalized_tracks.append({
            "id": track_id,
            "name": str(raw_track.get("name") or audio_path.stem),
            "audio_path": str(audio_path),
            "enabled": bool(raw_track.get("enabled", False)),
            "route": route,
            "status": str(raw_track.get("status") or COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.manual_midi.not_selected")),
            "midi_path": midi_path,
            "source": str(raw_track.get("source") or "separated"),
            "color": str(raw_track.get("color") or MIXER_TRACK_COLORS[0]),
        })
    return {"mode": mode, "request_dir": str(request_dir), "tracks": normalized_tracks}


def _new_track(track_id, name, audio_path, source="separated", color=None):
    return {
        "id": str(track_id), "name": str(name), "audio_path": str(audio_path),
        "enabled": False, "route": DEFAULT_MANUAL_MIDI_ROUTE,
        "status": COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.manual_midi.not_selected"), "midi_path": None, "source": source,
        "color": str(color or MIXER_TRACK_COLORS[0]),
    }


def _build_track_state(result, request_dir_value):
    if not isinstance(result, SeparationResult):
        raise TypeError("Expected SeparationResult")
    request_dir = _require_active_request_dir(request_dir_value)
    mode = result.mode
    if mode not in SPLIT_PROCESSING_MODES:
        raise ValueError(f"Unsupported separation result mode: {mode!r}")
    expected_keys = ("vocals", "accompaniment") if mode == "vocal_split" else tuple(STEM_KEYS)
    if tuple(result.separated_audio) != expected_keys:
        raise RuntimeError(f"Separated WAV order/set mismatch: expected={expected_keys}, actual={tuple(result.separated_audio)}")
    tracks = []
    for index, key in enumerate(expected_keys):
        path = _require_owned_request_file(request_dir, result.separated_audio[key], f"{key} separated WAV", {".wav"})
        tracks.append(_new_track(
            f"separated-{index}-{key}", key, path,
            color=MIXER_TRACK_COLORS[index % len(MIXER_TRACK_COLORS)],
        ))
    return _normalize_track_state({"mode": mode, "request_dir": str(request_dir), "tracks": tracks})


def _validate_processing_outputs(result, config, request_dir_value):
    if config.processing_mode not in DIRECT_PROCESSING_MODES:
        raise RuntimeError(f"Direct MIDI validation does not accept mode {config.processing_mode!r}")
    for attribute in (
        "separated_audio", "stem_midi_paths", "vocal_midi_path",
        "accompaniment_midi_path", "merged_midi_path",
    ):
        if getattr(result, attribute, None):
            raise RuntimeError(f"Direct mode unexpectedly returned split output: {attribute}")
    midi_path = _require_owned_request_file(request_dir_value, result.midi_path, "main MIDI", {".mid", ".midi"})
    return [str(midi_path)]


_ACTIVE_JOB_LOCK = threading.Lock()
_ACTIVE_JOB = None


def _register_active_job(job):
    global _ACTIVE_JOB
    with _ACTIVE_JOB_LOCK:
        _ACTIVE_JOB = job


def _unregister_active_job(job):
    global _ACTIVE_JOB
    if job is None:
        return
    with _ACTIVE_JOB_LOCK:
        if _ACTIVE_JOB is job:
            _ACTIVE_JOB = None


def request_stop_current_job():
    """Ask the running job to stop at the next cooperative checkpoint.

    Mirrors the desktop stop button: the pipeline and the separation
    service both poll their cancel flag between stages.
    """
    with _ACTIVE_JOB_LOCK:
        job = _ACTIVE_JOB
    if job is None:
        return gr.update()
    job.cancel()
    return COLAB_TRANSLATOR.t("status.cancelling")


def convert_audio_to_midi(audio_path, mode, backend_label, yourmt3_model_label, progress=gr.Progress()):
    source_audio = _require_source_audio(audio_path)
    if mode not in COLAB_MODE_IDS:
        raise gr.Error(ct("error.unknown_mode", mode=mode))
    backend_key = resolve_multi_backend(backend_label)
    yourmt3_model_key = YOURMT3_MODEL_LABEL_TO_KEY.get(yourmt3_model_label)
    if yourmt3_model_key is None:
        raise gr.Error(ct("error.unknown_yourmt3_model", model=yourmt3_model_label))

    config_kwargs = {"processing_mode": mode, "yourmt3_model": yourmt3_model_key}
    if mode == "smart":
        config_kwargs.update(transcription_backend=backend_key, multi_instrument_model=backend_key)
    config = Config(**config_kwargs)
    config.validate()

    clear_logs()
    logger.info("========== New Request ==========")
    logger.info(f"Audio file: {source_audio.name}")
    logger.info(f"Mode: {mode}")
    output_dir = Path(tempfile.mkdtemp(prefix="request-", dir=COLAB_OUTPUT_ROOT)).resolve()
    logger.info(f"Request dir: {output_dir}")
    on_progress = _make_progress_callback(progress)
    active_job = None
    try:
        prepare_selected_models(mode, backend_key, yourmt3_model_key)
        if mode in SPLIT_PROCESSING_MODES:
            active_job = AudioSeparationService(config, progress_callback=on_progress)
            _register_active_job(active_job)
            result = active_job.process(audio_path=source_audio, output_dir=output_dir)
            track_state = _build_track_state(result, output_dir)
            output_files = [track["audio_path"] for track in track_state["tracks"]]
            wav_lines = [
                f"  • {track['name']}: {Path(track['audio_path']).name}"
                for track in track_state["tracks"]
            ]
            status_lines = [
                COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.separation_result_title"),
                f"{COLAB_TRANSLATOR.t('dialogs.complete.audio_tracks.separation_mode')}: {COLAB_TRANSLATOR.t(f'main.mode.{mode}')}",
                f"{ct('status.elapsed')}: {result.processing_time:.1f} {ct('status.seconds')}",
                f"{COLAB_TRANSLATOR.t('dialogs.complete.stem_audio_count')}: {len(output_files)}",
                COLAB_TRANSLATOR.t("dialogs.complete.separated_audio") + ":",
                *wav_lines,
                COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.separation_manual_hint"),
            ]
            logger.info("========== Separation Finished ==========")
            return output_files, "\n".join(status_lines), track_state

        active_job = MusicToMidiPipeline(config)
        _register_active_job(active_job)
        result = active_job.process(
            audio_path=str(source_audio),
            output_dir=str(output_dir),
            progress_callback=on_progress,
        )
        output_files = _validate_processing_outputs(result, config, output_dir)
        bpm_str = f"{result.beat_info.bpm:.1f}" if result.beat_info else "N/A"
        status_lines = [
            ct("status.complete_header"),
            f"{ct('status.elapsed')}: {result.processing_time:.1f} {ct('status.seconds')}",
            f"{ct('status.total_notes')}: {result.total_notes}",
            f"{COLAB_TRANSLATOR.t('dialogs.complete.track_count')}: {len(result.tracks)}",
            f"BPM: {bpm_str}",
            f"{ct('status.device')}: {device_label}",
            f"{ct('status.midi_file')}: {Path(result.midi_path).name}",
        ]
        logger.info("========== Conversion Finished ==========")
        return output_files, "\n".join(status_lines), {}
    except InterruptedError:
        logger.info("Processing cancelled by user")
        try:
            shutil.rmtree(output_dir)
        except OSError as cleanup_exc:
            logger.error(f"Unable to remove cancelled request directory {output_dir}: {cleanup_exc}")
        return [], COLAB_TRANSLATOR.t("status.cancelled"), {}
    except Exception as exc:
        logger.error(f"Processing failed: {exc}")
        try:
            shutil.rmtree(output_dir)
        except OSError as cleanup_exc:
            logger.error(f"Unable to remove failed request directory {output_dir}: {cleanup_exc}")
        raise gr.Error(ct("error.conversion_failed", error=exc)) from exc
    finally:
        _unregister_active_job(active_job)
        try:
            clear_gpu_memory()
            logger.info("GPU memory cleaned")
        except Exception as exc:
            logger.warning(f"GPU memory cleanup warning: {exc}")


def _update_track_selection(state, track_index, enabled, route):
    normalized = _normalize_track_state(state)
    if not 0 <= track_index < len(normalized["tracks"]):
        raise gr.Error(ct("error.invalid_track_state", error="track index out of range"))
    if route not in MANUAL_MIDI_ROUTES:
        raise gr.Error(ct("error.unknown_manual_route", route=route))
    track = normalized["tracks"][track_index]
    track["enabled"] = bool(enabled)
    track["route"] = route
    track["status"] = (
        COLAB_TRANSLATOR.t(
            "dialogs.complete.audio_tracks.manual_midi.selected",
            model=_manual_route_display_label(route),
        )
        if enabled
        else COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.manual_midi.not_selected")
    )
    return normalized, track["status"]


def convert_track_to_midi(state, track_index, enabled, route, progress=gr.Progress()):
    normalized, _status = _update_track_selection(state, track_index, enabled, route)
    track = normalized["tracks"][track_index]
    if not track["enabled"]:
        raise gr.Error(ct("error.track_not_enabled"))
    request_dir = _require_active_request_dir(normalized["request_dir"])
    audio_path = _require_owned_request_file(request_dir, track["audio_path"], "selected track audio", SUPPORTED_AUDIO_SUFFIXES)
    pipeline = None
    try:
        config = prepare_manual_midi_route(route)
        output_dir = _require_owned_request_output_dir(request_dir, manual_midi_output_dir(audio_path, route))
        pipeline = MusicToMidiPipeline(config)
        _register_active_job(pipeline)
        result = pipeline.process(
            audio_path=str(audio_path),
            output_dir=str(output_dir),
            progress_callback=_make_progress_callback(progress),
        )
        midi_path = _require_owned_request_file(request_dir, result.midi_path, "track MIDI", {".mid", ".midi"})
        track["midi_path"] = str(midi_path)
        track["status"] = COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.manual_midi.complete", file=midi_path.name)
        return normalized, track["status"], str(midi_path)
    except InterruptedError:
        logger.info("Track conversion cancelled by user")
        track["status"] = COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.manual_midi.cancelled")
        return normalized, track["status"], None
    except Exception as exc:
        logger.error(f"Track conversion failed: {exc}")
        raise gr.Error(ct("error.conversion_failed", error=exc)) from exc
    finally:
        _unregister_active_job(pipeline)
        try:
            clear_gpu_memory()
        except Exception as exc:
            logger.warning(f"GPU memory cleanup warning: {exc}")


def add_audio_tracks(state, uploaded_files):
    normalized = _normalize_track_state(state)
    if not uploaded_files:
        raise gr.Error(ct("error.add_audio_required"))
    if isinstance(uploaded_files, (str, Path)):
        uploaded_files = [uploaded_files]
    if len(normalized["tracks"]) + len(uploaded_files) > MAX_MANUAL_TRACK_ROWS:
        raise gr.Error(ct("error.too_many_tracks", limit=MAX_MANUAL_TRACK_ROWS))
    request_dir = _require_active_request_dir(normalized["request_dir"])
    added_dir = request_dir / "added"
    added_dir.mkdir(parents=True, exist_ok=True)
    for uploaded in uploaded_files:
        source_value = getattr(uploaded, "name", uploaded)
        source = _require_source_audio(source_value)
        safe_name = Path(source.name).name
        destination = added_dir / safe_name
        if destination.exists():
            destination = added_dir / f"{destination.stem}-{uuid.uuid4().hex[:8]}{destination.suffix.lower()}"
        shutil.copy2(source, destination)
        copied = _require_owned_request_file(request_dir, destination, "added audio", SUPPORTED_AUDIO_SUFFIXES)
        track_id = f"added-{uuid.uuid4().hex}"
        normalized["tracks"].append(_new_track(
            track_id, copied.stem, copied, source="added",
            color=MIXER_TRACK_COLORS[len(normalized["tracks"]) % len(MIXER_TRACK_COLORS)],
        ))
    normalized = _normalize_track_state(normalized)
    return normalized, None, ct("status.audio_added", count=len(uploaded_files))


def update_mode_info(mode, backend_label=DEFAULT_MULTI_BACKEND_LABEL):
    if mode not in COLAB_MODE_IDS:
        raise gr.Error(ct("error.unknown_mode", mode=mode))
    terms = dict(COLAB_MODE_INFO_TERMS[mode])
    if mode == "smart":
        backend_key = resolve_multi_backend(backend_label)
        terms["backend_route"] = COLAB_BACKEND_ROUTE_NAMES[backend_key]
    return ct(f"mode_info.{mode}", **terms)


def update_yourmt3_model_info(yourmt3_model_label):
    model_key = YOURMT3_MODEL_LABEL_TO_KEY.get(yourmt3_model_label)
    if model_key is None:
        raise gr.Error(ct("error.unknown_yourmt3_model", model=yourmt3_model_label))
    model_info = YOURMT3_MODELS[model_key]
    label = model_info.get("ui_label", yourmt3_model_label)
    is_zh = COLAB_LANGUAGE.startswith("zh")
    if is_zh:
        description = model_info.get("description") or model_info.get("ui_description", "")
        feature_items = model_info.get("features_zh") or model_info.get("features", [])
        separator = "："
        features = "，".join(feature_items)
    else:
        description = model_info.get("ui_description") or model_info.get("description", "")
        feature_items = model_info.get("features_en") or model_info.get("features", [])
        separator = ": "
        features = ", ".join(feature_items)
    checkpoint = model_info.get("checkpoint", "")
    return (
        f"**{ct('yourmt3.model_title')}{separator}{label}**\n\n{description}\n\n"
        f"**{ct('yourmt3.checkpoint')}**{separator}`{checkpoint}`\n\n"
        f"**{ct('yourmt3.traits')}**{separator}{features}"
    )


def update_output_info(mode):
    if mode == "vocal_split":
        return f"**{ct('output.title')}**\n\n- {ct('output.vocal_manual')}\n- {ct('output.manual_tracks')}"
    if mode == "six_stem_split":
        return f"**{ct('output.title')}**\n\n- {ct('output.six_stem_manual')}\n- {ct('output.manual_tracks')}"
    return f"**{ct('output.title')}**\n\n- {ct('output.single_midi')}"


def update_mode_controls(mode, backend_label):
    if mode not in COLAB_MODE_IDS:
        raise gr.Error(ct("error.unknown_mode", mode=mode))
    backend_key = resolve_multi_backend(backend_label)
    is_smart = mode == "smart"
    uses_yourmt3 = is_smart and backend_key == "yourmt3"
    start_label = (
        "▶  " + COLAB_TRANSLATOR.t("toolbar.start_separation")
        if mode in SPLIT_PROCESSING_MODES
        else "▶  " + COLAB_TRANSLATOR.t("toolbar.start_convert")
    )
    return (
        update_mode_info(mode, backend_label), update_output_info(mode),
        gr.update(visible=is_smart), gr.update(visible=uses_yourmt3),
        gr.update(visible=uses_yourmt3), gr.update(value=start_label), {},
        gr.update(visible=False),
    )


def update_backend_controls(mode, backend_label):
    if mode not in COLAB_MODE_IDS:
        raise gr.Error(ct("error.unknown_mode", mode=mode))
    backend_key = resolve_multi_backend(backend_label)
    uses_yourmt3 = mode == "smart" and backend_key == "yourmt3"
    return update_mode_info(mode, backend_label), gr.update(visible=uses_yourmt3), gr.update(visible=uses_yourmt3)


def _render_track_rows(state):
    outputs = []
    if not state:
        outputs.append(gr.update(visible=False))
        outputs.append(gr.update(value="", visible=False))
        for _index in range(MAX_MANUAL_TRACK_ROWS):
            outputs.extend([
                gr.update(visible=False), gr.update(value=""),
                gr.update(value=False), gr.update(value=DEFAULT_MANUAL_MIDI_ROUTE),
                gr.update(value=""), gr.update(value=None),
            ])
        return outputs
    normalized = _normalize_track_state(state)
    outputs.append(gr.update(visible=True))
    outputs.append(gr.update(
        value=build_track_mixer_html(normalized["tracks"], COLAB_TRANSLATOR.t),
        visible=True,
    ))
    for index in range(MAX_MANUAL_TRACK_ROWS):
        if index < len(normalized["tracks"]):
            track = normalized["tracks"][index]
            outputs.extend([
                gr.update(visible=True),
                gr.update(
                    value=(
                        f"### ♪ <span style='color:{track['color']}'>"
                        f"{display_track_name(track['name'], COLAB_TRANSLATOR.t)}</span>"
                    )
                ),
                gr.update(value=track["enabled"]),
                gr.update(value=track["route"]), gr.update(value=track["status"]),
                gr.update(value=track["midi_path"]),
            ])
        else:
            outputs.extend([
                gr.update(visible=False), gr.update(value=""),
                gr.update(value=False), gr.update(value=DEFAULT_MANUAL_MIDI_ROUTE),
                gr.update(value=""), gr.update(value=None),
            ])
    return outputs


def _make_track_selection_handler(track_index):
    def handler(state, enabled, route):
        return _update_track_selection(state, track_index, enabled, route)
    return handler


def _make_track_conversion_handler(track_index):
    def handler(state, enabled, route, progress=gr.Progress()):
        return convert_track_to_midi(state, track_index, enabled, route, progress)
    return handler


def _make_track_remove_handler(track_index):
    def handler(state):
        normalized = _normalize_track_state(state)
        tracks = list(normalized["tracks"])
        if not 0 <= track_index < len(tracks):
            raise gr.Error(ct("error.invalid_track_state", error="track index out of range"))
        del tracks[track_index]
        return {**normalized, "tracks": tracks}
    return handler

LOG_POLL_JS = """<script>
(function() {
    var _pollTimer = setInterval(function() {
        var ta = document.querySelector('.log-box textarea');
        if (!ta) return;
        var setter = Object.getOwnPropertyDescriptor(HTMLTextAreaElement.prototype, 'value').set;
        fetch('./api/read_logs', {method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify({data: []})})
        .then(function(r) { return r.json(); })
        .then(function(json) {
            var logText = (json.data && json.data[0]) ? json.data[0] : '';
            if (logText) setter.call(ta, logText);
            ta.dispatchEvent(new Event('input', {bubbles: true}));
            ta.scrollTop = ta.scrollHeight;
        })
        .catch(function() {});
    }, 1200);
})();
</script>"""

with gr.Blocks(
    title="Music to MIDI (Colab GPU)",
    head=LOG_POLL_JS + mixer_head(),
    delete_cache=(3600, 86400),
    theme=gr.themes.Base(primary_hue=gr.themes.colors.blue, neutral_hue=gr.themes.colors.slate),
) as demo:
    gr.Markdown(ct("ui.header"))
    track_state = gr.State({})

    with gr.Row():
        with gr.Column(scale=5):
            audio_input = gr.Audio(label=COLAB_TRANSLATOR.t("space.ui.audio_input"), type="filepath")
            gr.Markdown(f"<small>{ct('ui.audio_hint')}</small>")
            mode_radio = gr.Radio(COLAB_MODE_CHOICES, value="smart", label=ct("ui.mode_label"))
            mode_info = gr.Markdown(update_mode_info("smart", DEFAULT_MULTI_BACKEND_LABEL))
            backend_dropdown = gr.Dropdown(
                choices=MULTI_BACKEND_CHOICES,
                value=DEFAULT_MULTI_BACKEND_LABEL,
                label=COLAB_TRANSLATOR.t("main.engine.active_label"),
                info=ct("ui.backend_info"),
            )
            yourmt3_model_dropdown = gr.Dropdown(
                choices=YOURMT3_MODEL_CHOICES,
                value=DEFAULT_YOURMT3_MODEL_LABEL,
                label=COLAB_TRANSLATOR.t("main.engine.yourmt3_model_label"),
                info=ct("ui.yourmt3_model_info"),
            )
            yourmt3_model_info = gr.Markdown(update_yourmt3_model_info(DEFAULT_YOURMT3_MODEL_LABEL))
            output_info = gr.Markdown(update_output_info("smart"))
            with gr.Row():
                convert_btn = gr.Button(
                    "▶  " + COLAB_TRANSLATOR.t("toolbar.start_convert"),
                    variant="primary", size="lg", scale=4,
                )
                stop_btn = gr.Button(
                    "■  " + COLAB_TRANSLATOR.t("toolbar.stop"),
                    variant="stop", size="lg", scale=1,
                )
            gr.Markdown(f"{ct('ui.device')}: **{device_label}**")

        with gr.Column(scale=5):
            status_output = gr.Textbox(
                label=ct("ui.status"), interactive=False, lines=7,
                placeholder=ct("ui.status_placeholder"),
            )
            file_output = gr.File(label=COLAB_TRANSLATOR.t("space.ui.download_label"), file_count="multiple")
            log_output = gr.Textbox(
                label=COLAB_TRANSLATOR.t("space.ui.logs_label"), interactive=False, lines=16,
                max_lines=30, elem_classes="log-box",
            )

    with gr.Group(visible=False) as timeline_group:
        gr.Markdown(f"## {COLAB_TRANSLATOR.t('dialogs.complete.audio_tracks.title')}")
        gr.Markdown(COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.subtitle"))
        with gr.Row():
            added_audio_input = gr.File(
                label=COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.add_track"), file_count="multiple", type="filepath",
                file_types=["audio"],
            )
            add_audio_btn = gr.Button(COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.add_track"))
        add_audio_status = gr.Textbox(label=ct("ui.status"), interactive=False)
        mixer_html = gr.HTML(value="", visible=False)

        track_groups = []
        track_titles = []
        track_enable_checks = []
        track_route_dropdowns = []
        track_status_boxes = []
        track_midi_files = []
        track_start_buttons = []
        track_remove_buttons = []
        for row_index in range(MAX_MANUAL_TRACK_ROWS):
            with gr.Group(visible=False) as track_group:
                with gr.Row():
                    track_title = gr.Markdown()
                    track_remove = gr.Button(
                        COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.remove"),
                        variant="stop", size="sm", scale=0,
                    )
                with gr.Row():
                    track_enable = gr.Checkbox(value=False, label=ct("ui.track_enable"))
                    track_route = gr.Dropdown(
                        choices=MANUAL_MIDI_ROUTE_CHOICES,
                        value=DEFAULT_MANUAL_MIDI_ROUTE,
                        label=COLAB_TRANSLATOR.t("dialogs.complete.audio_tracks.manual_midi.select_model"),
                    )
                    track_start = gr.Button(ct("ui.track_start"), variant="primary")
                track_status = gr.Textbox(label=ct("ui.track_status"), interactive=False)
                track_midi = gr.File(label=ct("ui.track_download"), file_count="single")
            track_groups.append(track_group)
            track_titles.append(track_title)
            track_enable_checks.append(track_enable)
            track_route_dropdowns.append(track_route)
            track_status_boxes.append(track_status)
            track_midi_files.append(track_midi)
            track_start_buttons.append(track_start)
            track_remove_buttons.append(track_remove)

    track_render_outputs = [timeline_group, mixer_html]
    for row_index in range(MAX_MANUAL_TRACK_ROWS):
        track_render_outputs.extend([
            track_groups[row_index], track_titles[row_index],
            track_enable_checks[row_index], track_route_dropdowns[row_index],
            track_status_boxes[row_index], track_midi_files[row_index],
        ])

    mode_radio.change(
        fn=update_mode_controls,
        inputs=[mode_radio, backend_dropdown],
        outputs=[
            mode_info, output_info, backend_dropdown, yourmt3_model_dropdown,
            yourmt3_model_info, convert_btn, track_state, timeline_group,
        ],
        api_name=False,
    )
    backend_dropdown.change(
        fn=update_backend_controls,
        inputs=[mode_radio, backend_dropdown],
        outputs=[mode_info, yourmt3_model_dropdown, yourmt3_model_info],
        api_name=False,
    )
    yourmt3_model_dropdown.change(
        fn=update_yourmt3_model_info,
        inputs=[yourmt3_model_dropdown],
        outputs=[yourmt3_model_info],
        api_name=False,
    )
    convert_event = convert_btn.click(
        fn=convert_audio_to_midi,
        inputs=[audio_input, mode_radio, backend_dropdown, yourmt3_model_dropdown],
        outputs=[file_output, status_output, track_state],
    )
    convert_event.then(
        fn=_render_track_rows,
        inputs=[track_state],
        outputs=track_render_outputs,
        api_name=False,
    )
    stop_btn.click(
        fn=request_stop_current_job,
        inputs=None,
        outputs=[status_output],
        api_name=False,
        queue=False,
    )
    add_event = add_audio_btn.click(
        fn=add_audio_tracks,
        inputs=[track_state, added_audio_input],
        outputs=[track_state, added_audio_input, add_audio_status],
    )
    add_event.then(
        fn=_render_track_rows,
        inputs=[track_state],
        outputs=track_render_outputs,
        api_name=False,
    )
    for row_index in range(MAX_MANUAL_TRACK_ROWS):
        selection_handler = _make_track_selection_handler(row_index)
        track_enable_checks[row_index].change(
            fn=selection_handler,
            inputs=[track_state, track_enable_checks[row_index], track_route_dropdowns[row_index]],
            outputs=[track_state, track_status_boxes[row_index]],
            api_name=False,
        )
        track_route_dropdowns[row_index].change(
            fn=selection_handler,
            inputs=[track_state, track_enable_checks[row_index], track_route_dropdowns[row_index]],
            outputs=[track_state, track_status_boxes[row_index]],
            api_name=False,
        )
        track_event = track_start_buttons[row_index].click(
            fn=_make_track_conversion_handler(row_index),
            inputs=[track_state, track_enable_checks[row_index], track_route_dropdowns[row_index]],
            outputs=[track_state, track_status_boxes[row_index], track_midi_files[row_index]],
        )
        track_event.then(
            fn=_render_track_rows,
            inputs=[track_state],
            outputs=track_render_outputs,
            api_name=False,
        )
        remove_event = track_remove_buttons[row_index].click(
            fn=_make_track_remove_handler(row_index),
            inputs=[track_state],
            outputs=[track_state],
            api_name=False,
        )
        remove_event.then(
            fn=_render_track_rows,
            inputs=[track_state],
            outputs=track_render_outputs,
            api_name=False,
        )

    _log_poll_btn = gr.Button(visible=False)
    _log_poll_btn.click(fn=read_logs, inputs=[], outputs=[log_output], api_name="read_logs", queue=False)
    gr.Markdown(ct("ui.footer"))

print("\n" + "=" * 60)
print(ct("ui.launching"))
print("=" * 60 + "\n")
demo.launch(share=True, server_name="0.0.0.0", server_port=7860, allowed_paths=[str(COLAB_OUTPUT_ROOT)])
